# Notebook 02 - Embeddings, Tokenizacion y Self-Attention

## Objetivos
- Usar tokenizadores de Hugging Face sobre texto real.
- Relacionar tokens con vectores de embedding.
- Implementar Scaled Dot-Product Attention desde cero con PyTorch.

## Introduccion
La Self-Attention es el nucleo del Transformer. Antes de cargar modelos preentrenados, conviene entender como se calculan Q, K y V y como se obtienen pesos de atencion normalizados.

In [3]:
# Importamos PyTorch y utilidades de tokenizacion
from pathlib import Path
from IPython.display import display
import torch
import torch.nn.functional as F
import pandas as pd
from transformers import AutoTokenizer

print('PyTorch version:', torch.__version__)

c:\Users\juand\anaconda3\envs\tf_windows\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


PyTorch version: 2.8.0+cpu


## 1) Tokenizacion con Hugging Face

In [4]:
# Cargamos un tokenizador simple (sin modelo pesado)
tokenizer = AutoTokenizer.from_pretrained('distilbert-base-uncased')

frase_ejemplo = 'Los transformers revolucionaron el NLP'
# Convertimos texto a IDs de tokens
encoding = tokenizer(frase_ejemplo, return_tensors='pt')
print('Tokens:', tokenizer.convert_ids_to_tokens(encoding['input_ids'][0]))
print('input_ids shape:', encoding['input_ids'].shape)

c:\Users\juand\anaconda3\envs\tf_windows\lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\juand\.cache\huggingface\hub\models--distilbert-base-uncased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


Tokens: ['[CLS]', 'los', 'transformers', 'rev', '##ol', '##uc', '##ion', '##aro', '##n', 'el', 'nl', '##p', '[SEP]']
input_ids shape: torch.Size([1, 13])


## 2) Embeddings aleatorios para demostracion

In [5]:
# Definimos dimension de embedding y vocabulario pequeno
vocab_size = tokenizer.vocab_size
embed_dim = 64

# Capa de embedding inicializada aleatoriamente (solo didactica)
embedding_layer = torch.nn.Embedding(vocab_size, embed_dim)

# Obtenemos vectores para la frase tokenizada
embeds = embedding_layer(encoding['input_ids'])
print('Shape embeddings:', embeds.shape)  # (batch, seq_len, embed_dim)

Shape embeddings: torch.Size([1, 13, 64])


## 3) Proyecciones Q, K y V

In [6]:
# Creamos matrices de proyeccion lineales para Q, K y V
d_model = embed_dim
W_q = torch.nn.Linear(d_model, d_model, bias=False)
W_k = torch.nn.Linear(d_model, d_model, bias=False)
W_v = torch.nn.Linear(d_model, d_model, bias=False)

# Calculamos consultas, claves y valores
Q = W_q(embeds)
K = W_k(embeds)
V = W_v(embeds)
print('Shapes Q/K/V:', Q.shape, K.shape, V.shape)

Shapes Q/K/V: torch.Size([1, 13, 64]) torch.Size([1, 13, 64]) torch.Size([1, 13, 64])


## 4) Scaled Dot-Product Attention desde cero

In [7]:
def scaled_dot_product_attention(Q, K, V, mask=None):
    """Implementacion didactica de atencion escalada."""
    # d_k es la dimension de las claves
    d_k = K.size(-1)
    # Puntajes = QK^T dividido por sqrt(d_k)
    scores = torch.matmul(Q, K.transpose(-2, -1)) / (d_k ** 0.5)
    if mask is not None:
        scores = scores.masked_fill(mask == 0, float('-inf'))
    # Softmax sobre la ultima dimension
    weights = F.softmax(scores, dim=-1)
    # Salida ponderada por V
    output = torch.matmul(weights, V)
    return output, weights

salida, pesos = scaled_dot_product_attention(Q, K, V)
print('Salida atencion shape:', salida.shape)
print('Matriz de pesos shape:', pesos.shape)

Salida atencion shape: torch.Size([1, 13, 64])
Matriz de pesos shape: torch.Size([1, 13, 13])


## 5) Intuicion Q/K/V con 12 frases de ejemplo

In [8]:
# Lista de frases para explorar tokenizacion y longitudes
frases_qkv = [
    "El gato persigue al raton",
    "María le dijo a Ana que ella ganó",
    "Los transformers usan atencion",
    "Python es un lenguaje de programacion",
    "La self-attention calcula pesos",
    "El perro ladra en el jardin",
    "OpenAI publico GPT",
    "BERT enmascara tokens aleatorios",
    "La traduccion automatica mejoro con atencion",
    "El token CLS resume la frase",
    "Los embeddings capturan semantica",
    "El modelo aprende relaciones contextuales"
]

resumen = []
for frase in frases_qkv:
    ids = tokenizer(frase, return_tensors='pt')['input_ids']
    tokens = tokenizer.convert_ids_to_tokens(ids[0])
    resumen.append({
        'frase': frase,
        'num_tokens': len(tokens),
        'tokens': ' '.join(tokens),
    })

df_qkv = pd.DataFrame(resumen)
display(df_qkv)

,frase,num_tokens,tokens
0,El gato persigue al raton,11,[CLS] el ga ##to per ##si ##gue al rat ##on [SEP]
1,María le dijo a Ana que ella ganó,12,[CLS] maria le di ##jo a ana que ella gan ##o ...
2,Los transformers usan atencion,9,[CLS] los transformers usa ##n ate ##nc ##ion ...
3,Python es un lenguaje de programacion,11,[CLS] python es un len ##gua ##je de program #...
4,La self-attention calcula pesos,10,[CLS] la self - attention cal ##cula pe ##sos ...
5,El perro ladra en el jardin,11,[CLS] el per ##ro lad ##ra en el jar ##din [SEP]
6,OpenAI publico GPT,8,[CLS] open ##ai public ##o gp ##t [SEP]
7,BERT enmascara tokens aleatorios,12,[CLS] bert en ##mas ##car ##a token ##s ale ##...
8,La traduccion automatica mejoro con atencion,16,[CLS] la tr ##ad ##ucci ##on automatic ##a me ...
9,El token CLS resume la frase,10,[CLS] el token cl ##s resume la fra ##se [SEP]


## 6) Interpretacion de pesos de atencion en una frase

In [9]:
# Tomamos una frase corta y mostramos pesos token-token
frase_corta = 'El gato persigue al raton'
enc = tokenizer(frase_corta, return_tensors='pt')
emb = embedding_layer(enc['input_ids'])
q, k, v = W_q(emb), W_k(emb), W_v(emb)
_, attn = scaled_dot_product_attention(q, k, v)

tokens = tokenizer.convert_ids_to_tokens(enc['input_ids'][0])
matriz = pd.DataFrame(attn[0].detach().numpy(), index=tokens, columns=tokens)
display(matriz.round(3))

,[CLS],el,ga,##to,per,##si,##gue,al,rat,##on,[SEP]
[CLS],0.060,0.075,0.062,0.117,0.088,0.092,0.164,0.125,0.080,0.063,0.074
el,0.071,0.073,0.112,0.117,0.095,0.133,0.088,0.104,0.048,0.066,0.092
ga,0.070,0.115,0.104,0.084,0.112,0.109,0.073,0.089,0.084,0.075,0.084
##to,0.085,0.089,0.096,0.063,0.125,0.097,0.048,0.156,0.121,0.054,0.065
per,0.067,0.055,0.108,0.125,0.070,0.086,0.141,0.116,0.053,0.110,0.068
##si,0.085,0.082,0.092,0.100,0.092,0.068,0.077,0.155,0.097,0.069,0.084
##gue,0.110,0.067,0.105,0.047,0.107,0.112,0.077,0.081,0.080,0.146,0.068
al,0.056,0.094,0.102,0.086,0.081,0.096,0.050,0.083,0.139,0.120,0.094
rat,0.118,0.093,0.057,0.050,0.153,0.070,0.064,0.067,0.064,0.115,0.148
##on,0.091,0.074,0.182,0.085,0.074,0.094,0.075,0.065,0.058,0.115,0.086


## Resultados
Tokenizamos texto, generamos embeddings, proyectamos Q/K/V e implementamos atencion escalada sin modelos preentrenados.

## Conclusiones
La Self-Attention permite que cada token consulte a todos los demas. Escalar por sqrt(d_k) estabiliza los gradientes cuando la dimension crece.

## Ejercicios guiados resueltos
**Ejercicio:** Calcula la suma por fila de la matriz de atencion y verifica que sea 1.

**Solucion:**

In [ ]:
sumas_fila = attn[0].sum(dim=-1)
print('Sumas por fila (deben ser ~1):', sumas_fila.detach().numpy().round(4))

## Ejercicios propuestos
1. Implementa una mascara causal para un decoder.
2. Compara BPE vs WordPiece en una misma frase.
3. Visualiza heatmap de `matriz` con seaborn.

## Preguntas de reflexion
1. Por que Q, K y V usan proyecciones distintas?
2. Que ocurre si omites la division por sqrt(d_k)?
3. Como cambia el numero de tokens al usar subpalabras?